<a href="https://colab.research.google.com/github/efitznm/ds-toolbox/blob/main/DS_Cheat_Sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile metadata01.py
import pandas as pd

def metadata( dataframe ):
  '''Given a dataframe, returns a dataframe of metadata about the dataframe'''
  metadata_df = pd.DataFrame()
  metadata_df["Data_types"] = dataframe.dtypes
  metadata_df["Count"] = len(dataframe)
  metadata_df["Nulls"] = dataframe.isnull().sum()
  metadata_df["Nulls_pct"] = ( metadata_df["Nulls"] / metadata_df["Count"] * 100 ).round(1)
  metadata_df["Memory"] = dataframe.memory_usage( deep = True)
  metadata_df["NUnique"] = dataframe.nunique()
  metadata_df["NUnique_pct"] = (dataframe.nunique() / metadata_df["Count"] * 100).round(1)
  metadata_df = metadata_df.join( dataframe.describe( include = "all" ).transpose() )
  metadata_df = metadata_df.astype( { "count" : int } )
  if dataframe.select_dtypes(include=['number']).shape[1] :
    metadata_df["IRQ"] = metadata_df["75%"] - metadata_df["25%"]
    metadata_df["range"] = metadata_df["max"] - metadata_df["min"]
    metadata_df["sum"] = metadata_df["mean"] * metadata_df["count"]
    metadata_df = (
      metadata_df
      .rename( columns = {
        "25%" : "Q1_25%",
        "50%" : "Q2_median",
        "75%" : "Q3_75%",
        }
      )
    )
  return metadata_df

def cols_to_drop( dataframe ):
  '''Given a dataframe, returns columns that should likely be dropped'''
  md = metadata( dataframe )
  filter = md["Nulls_pct"] >= 40
  return md[ filter ]["Nulls_pct"].to_dict()

def likely_ids( dataframe ):
  '''Given a dataframe, returns a dictionary of likely ID columns'''
  md = metadata( dataframe )
  return md["NUnique_pct"][ md["NUnique_pct"] > 95 ].sort_values( ascending = False )


Overwriting metadata01.py


In [ ]:
import metadata01 as md01
import pandas as pd

Transportation Equipment Manufacturing Data

In [ ]:
url = 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/Equipment.csv'
df = pd.read_csv( url )

In [ ]:
! curl -I {url}

HTTP/2 200 
server: GitHub.com
content-type: text/csv; charset=utf-8
last-modified: Mon, 15 Jun 2026 19:27:56 GMT
access-control-allow-origin: *
etag: "6a30523c-37c"
expires: Wed, 11 Nov 2026 15:35:00 GMT
cache-control: max-age=600
x-proxy-cache: MISS
x-github-request-id: 173A:112BEE:1E8878:25DA66:6AF488CC
x-github-edge-region: iad
accept-ranges: bytes
age: 0
date: Wed, 11 Nov 2026 15:58:51 GMT
via: 1.1 varnish
x-served-by: cache-pdk-kfty8610045-PDK
x-cache: HIT
x-cache-hits: 0
x-timer: S1794412731.353941,VS0,VE27
vary: Accept-Encoding
x-fastly-request-id: 3d6b8214223cdc56e069e22bfbb171ff6457534a
content-length: 892



In [ ]:
md01.metadata(df)

,Data_types,Count,Nulls,Nulls_pct,Memory,NUnique,NUnique_pct,count,unique,top,...,mean,std,min,Q1_25%,Q2_median,Q3_75%,max,IRQ,range,sum
rownames,object,25,0,0.0,1427,25,100.0,25,25,Alabama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
valueadded,float64,25,0,0.0,200,25,100.0,25,NaN,NaN,...,711.56,966.095883,22.7,126.148,494.515,690.67,4079.554,564.522,4056.854,17789.0
capital,float64,25,0,0.0,200,25,100.0,25,NaN,NaN,...,53.06308,93.01841,1.543,6.547,22.001,39.712,435.105,33.165,433.562,1326.577
labor,float64,25,0,0.0,200,25,100.0,25,NaN,NaN,...,105.58888,125.962605,4.063,31.301,83.033,109.728,490.384,78.427,486.321,2639.722
firms,int64,25,0,0.0,200,25,100.0,25,NaN,NaN,...,235.88,272.641297,15.0,81.0,154.0,275.0,1372.0,194.0,1357.0,5897.0


In [ ]:
df.shape

(25, 5)

In [ ]:
df.head()

,rownames,valueadded,capital,labor,firms
0,Alabama,126.148,3.804,31.551,68
1,California,3201.486,185.446,452.844,1372
2,Connecticut,690.670,39.712,124.074,154
3,Florida,56.296,6.547,19.181,292
4,Georgia,304.531,11.530,45.534,71


In [ ]:
md01.cols_to_drop(df)

{}

Actually 25 rows and 5 columns (says 4 columns on the webpage)

4 columns are numeric